In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import argparse
import os
import sys
from huggingface_hub import hf_hub_download
from src.feature_generation import extract_feature, extract_encoder
from src.preprocessing import BioacousticDataset
import time
import numpy as np
from pathlib import Path

/idiap/temp/adeych/conda/envs/temp-env-4/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import onnxruntime as ort

print("Available Providers:", ort.get_available_providers())

Available Providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [3]:
import torch

print("PyTorch CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))
print(torch.version.cuda)

PyTorch CUDA available: True
Device name: NVIDIA GeForce RTX 3090
13.0


In [5]:
dir = Path("/idiap/temp/adeych/data")
batdata = BioacousticDataset(data_input=str(dir / "bat_metadata.csv"),
                             root_dir=str(dir / "xenocanto-dataset"),
                             encoder = "perch2")

In [8]:
from src.feature_generation import build_perch_fb_io
X_bags, labels = build_perch_fb_io(batdata)

Dataset type: <class 'src.preprocessing.BioacousticDataset'>


2026-08-19 09:22:57.948196529 [W:onnxruntime:, transformer_memcpy.cc:111 ApplyImpl] 2 Memcpy nodes are added to the graph perch_v2 for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2026-08-19 09:22:57.996864808 [E:onnxruntime:, inference_session.cc:2879 operator()] Exception during initialization: /onnxruntime_src/onnxruntime/core/framework/bfc_arena.cc:359 void* onnxruntime::BFCArena::AllocateRawInternal(size_t, bool, onnxruntime::Stream*) Failed to allocate memory for requested buffer of size 363601920



Fail: [ONNXRuntimeError] : 1 : FAIL : Exception during initialization: /onnxruntime_src/onnxruntime/core/framework/bfc_arena.cc:359 void* onnxruntime::BFCArena::AllocateRawInternal(size_t, bool, onnxruntime::Stream*) Failed to allocate memory for requested buffer of size 363601920


In [6]:
from src.feature_generation import build_perch_fb
X_bags, labels = build_perch_fb(batdata,device = 'cuda') #30s 10%, 22% 1min, 50% 2min28,

Dataset type: <class 'src.preprocessing.BioacousticDataset'>


2026-08-19 09:26:56.553116904 [W:onnxruntime:, transformer_memcpy.cc:111 ApplyImpl] 2 Memcpy nodes are added to the graph perch_v2 for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
Extracting perch 2.0: 100%|██████████| 285/285 [05:41<00:00,  1.20s/it]


In [8]:
print(len(X_bags), labels.shape)

285 (285, 5)


In [9]:
print(X_bags[0].shape)

(104, 16, 4, 1536)


In [10]:
import pickle
pickle_filename = 'perch2-bags.pkl'
with open(pickle_filename, 'wb') as pickle_file:
    pickle.dump(X_bags, pickle_file)
np.save('perch2-labels.npy', labels)

In [11]:
labels_test_1 = np.load('perch2-labels.npy')
print(labels_test_1 == labels)

[[ True  True  True  True  True]
 [ True  True  True  True  True]
 [ True  True  True  True  True]
 ...
 [ True  True  True  True  True]
 [ True  True  True  True  True]
 [ True  True  True  True  True]]


In [12]:
with open('perch2-bags.pkl', 'rb') as pickle_file:
    X_bags_test_1 = pickle.load(pickle_file)
print(X_bags_test_1[0].shape)

(104, 16, 4, 1536)


In [13]:
X_per = X_bags
y_label = labels

In [10]:
from src.feature_generation import build_perch_fb
X_bags, labels = build_perch_fb(batdata) #1 min 10%

Dataset type: <class 'src.preprocessing.BioacousticDataset'>


Extracting perch 2.0:   9%|▉         | 25/285 [00:59<10:17,  2.37s/it]


KeyboardInterrupt: 

In [4]:
from src.feature_generation import build_feature_bank

encoder_name = "NLM_BEATs"
dir = Path("/idiap/temp/adeych/data")
batdata = BioacousticDataset(data_input=str(dir / "bat_metadata.csv"),
                             root_dir=str(dir / "xenocanto-dataset"),
                             encoder = encoder_name)


In [6]:
from avex import load_model,list_models

In [7]:
print(list_models())


Model Name                          Description                              Trained Classifier  
esp_aves2_eat_all                   eat_hf (pretrained backbone)             ❌ No                
esp_aves2_eat_bio                   eat_hf (pretrained backbone)             ❌ No                
esp_aves2_effnetb0_all              efficientnet (fine-tuned) - 12806 classes ✅ Yes (12806 classes)
esp_aves2_effnetb0_audioset         efficientnet (pretrained backbone)       ❌ No                
esp_aves2_effnetb0_bio              efficientnet (fine-tuned) - 12279 classes ✅ Yes (12279 classes)
esp_aves2_naturelm_audio_v1_beats   beats (pretrained backbone) - NatureLM   ❌ No                
esp_aves2_sl_beats_all              beats (fine-tuned) - 12806 classes - fine-tuned ✅ Yes (12806 classes)
esp_aves2_sl_beats_bio              beats (fine-tuned) - 12279 classes - fine-tuned ✅ Yes (12279 classes)
esp_aves2_sl_eat_all_ssl_all        eat_hf (fine-tuned) - 12806 classes      ✅ Yes (12806 classes

In [8]:
model = load_model("esp_aves2_naturelm_audio_v1_beats",return_features_only=True,device="cpu")

Error while downloading from https://huggingface.co/EarthSpeciesProject/esp-aves2-naturelm-audio-v1-beats/resolve/main/esp-aves2-naturelm-audio-v1-beats.safetensors: The read operation timed out
Trying to resume download...


In [5]:
encoder = extract_encoder(encoder_name, device='cpu')

KeyboardInterrupt: 

In [ ]:
X_NLM,labels = build_feature_bank(batdata, model, encoder_name, device='cpu')
#6% in 1 min

Dataset type: <class 'src.preprocessing.BioacousticDataset'>


Extracting NLM_BEATs:   0%|          | 0/285 [00:00<?, ?it/s]

Extracting NLM_BEATs:   6%|▌         | 16/285 [01:03<17:54,  4.00s/it] 


KeyboardInterrupt: 

In [11]:
encoder_cuda = extract_encoder(encoder_name, device='cuda')

In [ ]:
X_NLM,labels = build_feature_bank(batdata, encoder_cuda, encoder_name, device='cuda')
#25% in 30s, 60% in 1min, 80% in 1min40, 100% in 2min14

Dataset type: <class 'src.preprocessing.BioacousticDataset'>


Extracting NLM_BEATs: 100%|██████████| 285/285 [02:13<00:00,  2.14it/s]


In [ ]:
print

In [13]:
import pickle
pickle_filename = 'NLM-bags.pkl'
with open(pickle_filename, 'wb') as pickle_file:
    pickle.dump(X_NLM, pickle_file)

OSError: [Errno 122] Disk quota exceeded

In [15]:
import pickle
from pathlib import Path

# 1. Define folder and file path
folder_path = Path("/idiap/temp/adeych/data/feature_banks")
file_path = folder_path / "NLM-bags.pkl"

# 2. Ensure folder exists (creates it if it doesn't)
folder_path.mkdir(parents=True, exist_ok=True)

# 3. Save your object
with open(file_path, "wb") as f:
    pickle.dump(X_NLM, f)

print(f"Saved successfully to: {file_path}")

Saved successfully to: /idiap/temp/adeych/data/feature_banks/NLM-bags.pkl


In [16]:
print(X_NLM[0].shape)

(104, 496, 768)


In [17]:
encoder_name = "effnetb0"
dir = Path("/idiap/temp/adeych/data")
batdata = BioacousticDataset(data_input=str(dir / "bat_metadata.csv"),
                             root_dir=str(dir / "xenocanto-dataset"),
                             encoder = encoder_name)


In [18]:
encoder = extract_encoder(encoder_name, device='cuda')

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /idiap/home/adeych/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 166MB/s]


In [ ]:
X_eff,labels = build_feature_bank(batdata, encoder, encoder_name, device='cuda')
#25% in 10s, 50% in 30s, 100% in 1min8

Dataset type: <class 'src.preprocessing.BioacousticDataset'>


Extracting effnetb0:   0%|          | 0/285 [00:00<?, ?it/s]

Extracting effnetb0: 100%|██████████| 285/285 [01:08<00:00,  4.17it/s]


In [22]:
# 1. Define folder and file path
folder_path = Path("/idiap/temp/adeych/data/feature_banks")
file_path = folder_path / "effnetb0-bags.pkl"

# 2. Ensure folder exists (creates it if it doesn't)
folder_path.mkdir(parents=True, exist_ok=True)

# 3. Save your object
with open(file_path, "wb") as f:
    pickle.dump(X_eff, f)

print(f"Saved successfully to: {file_path}")

KeyboardInterrupt: 

In [23]:
import tensorflow as tf
import tensorflow_hub as hub

/idiap/temp/adeych/conda/envs/temp-env-4/lib/python3.12/site-packages/IPython/extensions/autoreload.py:367: RuntimeWarning: changes to cyfunction.__defaults__ will not currently affect the values used in function calls
  setattr(old, name, getattr(new, name))
/idiap/temp/adeych/conda/envs/temp-env-4/lib/python3.12/site-packages/IPython/extensions/autoreload.py:367: RuntimeWarning: changes to cyfunction.__kwdefaults__ will not currently affect the values used in function calls
  setattr(old, name, getattr(new, name))
I0000 00:00:1787127824.188501 1518703 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787127829.911073 1518703 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


ModuleNotFoundError: No module named 'pkg_resources'